In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
from datasets import load_dataset

ds = load_dataset("Tobi-Bueck/customer-support-tickets",
                   data_files="dataset-tickets-multi-lang-4-20k.csv")
df = ds["train"].to_pandas()

df = df[df["language"] == "en"].copy()
df = df.dropna(subset=["body", "queue"])
texts = df["body"].tolist()
labels = df["queue"].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(len(X_train), len(X_val), len(X_test))

c:\Users\sande\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


8345 1788 1789


In [3]:
import os, urllib.request, zipfile

glove_url = "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip"
glove_zip_path = "glove.6B.zip"
glove_txt_path = "glove.6B.200d.txt"

if not os.path.exists(glove_txt_path):
    urllib.request.urlretrieve(glove_url, glove_zip_path)
    with zipfile.ZipFile(glove_zip_path, "r") as zip_ref:
        zip_ref.extract(glove_txt_path, ".")

In [4]:
glove_vectors = {}
with open(glove_txt_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.split()
        word = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        glove_vectors[word] = vector

print(f"Loaded {len(glove_vectors)} word vectors")

Loaded 400001 word vectors


In [5]:
tfidf_vectorizer = TfidfVectorizer()
train_tfidf = tfidf_vectorizer.fit_transform(X_train)
val_tfidf = tfidf_vectorizer.transform(X_val)
test_tfidf = tfidf_vectorizer.transform(X_test)

print(train_tfidf.shape)

(8345, 4751)


In [6]:
feature_names = tfidf_vectorizer.get_feature_names_out()
word_to_index = {word: i for i, word in enumerate(feature_names)}

def weighted_glove_vector(text, tfidf_row, glove_vectors, embedding_dim=200):
    words = text.lower().split()
    weighted_sum = np.zeros(embedding_dim)
    total_weight = 0.0

    for word in words:
        if word in glove_vectors and word in word_to_index:
            tfidf_score = tfidf_row[0, word_to_index[word]]
            if tfidf_score > 0:
                weighted_sum += tfidf_score * glove_vectors[word]
                total_weight += tfidf_score

    if total_weight > 0:
        return weighted_sum / total_weight
    return np.zeros(embedding_dim)

In [7]:
def build_features(texts, tfidf_matrix):
    return np.array([
        weighted_glove_vector(text, tfidf_matrix[i], glove_vectors)
        for i, text in enumerate(texts)
    ])

train_features = build_features(X_train, train_tfidf)
val_features = build_features(X_val, val_tfidf)
test_features = build_features(X_test, test_tfidf)

print(train_features.shape, val_features.shape, test_features.shape)

(8345, 200) (1788, 200) (1789, 200)


In [12]:
class EmbeddingClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.hidden_layer = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.output_layer = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.hidden_layer(x)
        x = self.activation(x)
        x = self.output_layer(x)
        return x

label_encoder = LabelEncoder()
train_labels = torch.tensor(label_encoder.fit_transform(y_train), dtype=torch.long)
val_labels = torch.tensor(label_encoder.transform(y_val), dtype=torch.long)
test_labels = torch.tensor(label_encoder.transform(y_test), dtype=torch.long)

train_x = torch.tensor(train_features, dtype=torch.float32)
val_x = torch.tensor(val_features, dtype=torch.float32)
test_x = torch.tensor(test_features, dtype=torch.float32)

model = EmbeddingClassifier(
    input_size=train_x.shape[1],
    hidden_size=128,
    num_classes=len(label_encoder.classes_),
)
print(model)

EmbeddingClassifier(
  (hidden_layer): Linear(in_features=200, out_features=128, bias=True)
  (activation): ReLU()
  (output_layer): Linear(in_features=128, out_features=10, bias=True)
)


In [13]:
torch.manual_seed(42)

train_batches = DataLoader(TensorDataset(train_x, train_labels), batch_size=64, shuffle=True)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

for epoch in range(50):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_batches:
        optimizer.zero_grad()
        scores = model(batch_x)
        loss = loss_function(scores, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        train_predictions = model(train_x).argmax(dim=1)

    train_accuracy = accuracy_score(train_labels, train_predictions)

    print(f"Epoch {epoch + 1}: loss = {total_loss / len(train_batches):.4f}, "
          f"train acc = {train_accuracy:.3f}")
    

Epoch 1: loss = 1.8501, train acc = 0.375
Epoch 2: loss = 1.7241, train acc = 0.384
Epoch 3: loss = 1.6833, train acc = 0.383
Epoch 4: loss = 1.6611, train acc = 0.395
Epoch 5: loss = 1.6470, train acc = 0.411
Epoch 6: loss = 1.6120, train acc = 0.420
Epoch 7: loss = 1.5893, train acc = 0.424
Epoch 8: loss = 1.5682, train acc = 0.435
Epoch 9: loss = 1.5559, train acc = 0.443
Epoch 10: loss = 1.5435, train acc = 0.450
Epoch 11: loss = 1.5121, train acc = 0.456
Epoch 12: loss = 1.4897, train acc = 0.469
Epoch 13: loss = 1.4710, train acc = 0.472
Epoch 14: loss = 1.4506, train acc = 0.484
Epoch 15: loss = 1.4328, train acc = 0.495
Epoch 16: loss = 1.4115, train acc = 0.490
Epoch 17: loss = 1.3990, train acc = 0.511
Epoch 18: loss = 1.3736, train acc = 0.503
Epoch 19: loss = 1.3558, train acc = 0.516
Epoch 20: loss = 1.3426, train acc = 0.530
Epoch 21: loss = 1.3323, train acc = 0.528
Epoch 22: loss = 1.3070, train acc = 0.526
Epoch 23: loss = 1.2906, train acc = 0.544
Epoch 24: loss = 1.2

In [14]:
model.eval()
with torch.no_grad():
    val_predictions = model(val_x).argmax(dim=1)

print(f"Val accuracy: {accuracy_score(val_labels, val_predictions):.3f}")
print(f"Val macro F1: {f1_score(val_labels, val_predictions, average='macro'):.3f}")
print(classification_report(val_labels, val_predictions, target_names=label_encoder.classes_))

Val accuracy: 0.385
Val macro F1: 0.311
                                 precision    recall  f1-score   support

           Billing and Payments       0.60      0.68      0.63       195
               Customer Service       0.36      0.20      0.26       279
                General Inquiry       0.27      0.16      0.20        25
                Human Resources       0.17      0.16      0.17        31
                     IT Support       0.26      0.29      0.27       208
                Product Support       0.33      0.34      0.33       334
          Returns and Exchanges       0.21      0.10      0.14        87
            Sales and Pre-Sales       0.15      0.16      0.16        50
Service Outages and Maintenance       0.52      0.43      0.47        67
              Technical Support       0.43      0.53      0.47       512

                       accuracy                           0.39      1788
                      macro avg       0.33      0.31      0.31      1788
         